Poniżej znajduje się kompletny kod do uruchomienia RAG z porównaniem różnych modeli LLM. Kod jest podzielony na sekcje.
Tutaj poniżej jest **dodawanie danych z plików PDF do bazy wektorowej.**

In [ ]:
MODEL_EMBEDDING_NAME = "sentence-transformers/all-mpnet-base-v2"

# ------------------------------------------ Embedding --------------------------------------------
from langchain_huggingface import HuggingFaceEmbeddings
import os
MODEL_SAVE_PATH = "model_embedding"
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': True, 'batch_size': 8}
def get_embeddings():
    print("--- Inicjalizacja modelu embeddingowego ---")
    return HuggingFaceEmbeddings(
        model_name=MODEL_EMBEDDING_NAME,
        cache_folder=MODEL_SAVE_PATH,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs,
        multi_process=False
    )


# ------------------------------------------ Chunkowanie --------------------------------------------

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    SentenceTransformersTokenTextSplitter
)
CHUNK_SIZE = 900
CHUNK_OVERLAP = 50

def create_optimized_splitter():
    """Splitter zoptymalizowany pod polski język akademicki"""
    return RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
        separators=["\n\n", "\n", ". ", "! ", "? ",  "; ",  ", ", " ",  ""]
    )

def create_token_splitter():
    """Splitter bazujący na tokenach - dokładniejszy"""
    return SentenceTransformersTokenTextSplitter(
        chunk_overlap=CHUNK_OVERLAP,
        tokens_per_chunk=256, 
        model_name=MODEL_EMBEDDING_NAME
    )

def smart_chunk_documents(docs):
    """Inteligentne chunkowanie z diagnostyką"""
    print("--- Chunkowanie dokumentów ---")
    if not docs:
        print("Brak dokumentów do przetworzenia")
        return []
    total_chars = sum(len(doc.page_content) for doc in docs)
    avg_doc_length = total_chars / len(docs)
    print(f"Łączna liczba dokumentów: {len(docs)}")
    print(f"Średnia długość dokumentu: {avg_doc_length:.0f} znaków")
    if avg_doc_length > 1000:
        print("✓ Wybieram strategię dla długich dokumentów akademickich")
        splitter = create_optimized_splitter()
    else:
        print("✓ Wybieram strategię token-based dla lepszej precyzji")
        splitter = create_token_splitter()

    chunks = splitter.split_documents(docs)
    if chunks:
        avg_chunk_size = sum(len(chunk.page_content) for chunk in chunks) / len(chunks)
        print(f"Utworzono {len(chunks)} fragmentów")
        print(f"Średnia długość fragmentu: {avg_chunk_size:.0f} znaków")
    else:
        raise ValueError("Błąd: Nie udało się utworzyć fragmentów dokumentów")
    return chunks

# -------------------- Dodawanie do bazy -----------------------
import chromadb
import os
import time
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader

# === KONFIGURACJA ŚCIEZEK ===
DB_DIR = "bazawektorowa_pdf"
os.makedirs(DB_DIR, exist_ok=True)

def init_chroma():
    os.makedirs(DB_DIR, exist_ok=True)
    return chromadb.PersistentClient(path=DB_DIR)

def check_collection_exists(client, collection_name):
    """Sprawdza czy kolekcja istnieje i zawiera dane"""
    try:
        collection = client.get_collection(collection_name)
        count = collection.count()
        if count > 0:
            print(f"✓ Kolekcja '{collection_name}' istnieje i zawiera {count} dokumentów")
            return True
        else:
            print(f"✗ Kolekcja '{collection_name}' istnieje ale jest pusta")
            return False
    except Exception as e:
        print(f"✗ Kolekcja '{collection_name}' nie istnieje")
        return False

def save_to_vectorstore(documents, embeddings, client, collection_name):
    """Zapis dokumentów do bazy wektorowej"""
    return Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        client=client,
        collection_name=collection_name,
    )

def load_vectorstore(client, collection_name, embeddings):
    print("--- Ładowanie bazy wektorowej ---")
    return Chroma(
        client=client,
        collection_name=collection_name,
        embedding_function=embeddings
    )

def load_pdf_documents(documents_path):
    """Ładowanie dokumentów PDF z folderu"""
    print("--- Ładowanie PDFów ---")
    docs = []
    if not os.path.exists(documents_path):
        return []
    for file in os.listdir(documents_path):
        if file.endswith(".pdf"):
            try:
                loader = PyPDFLoader(os.path.join(documents_path, file))
                docs.extend(loader.load())
                print(f"✓ Załadowano: {file}")
            except Exception as e:
                print(f"✗ Błąd z {file}: {e}")
    return docs



def index_documents_in_batches(splits, embeddings, client, collection_name, batch_size=50):
    """Indeksowanie z postępem"""
    print("--- Indeksowanie w ChromaDB ---")
    start_time = time.time()
    vectorstore = None

    for i in range(0, len(splits), batch_size):
        batch = splits[i:i + batch_size]
        vectorstore = save_to_vectorstore(batch, embeddings, client, collection_name)
        print(f"Zaindeksowano {min(i + batch_size, len(splits))}/{len(splits)} fragmentów")

    print(f"⏱ Czas indeksowania: {time.time() - start_time:.2f}s")
    return vectorstore



# ---- INICJALIZACJA RAG -----------

import json
import os
import time
import psutil 
from llama_cpp import Llama
from huggingface_hub import hf_hub_download
from tqdm import tqdm


# === KONFIGURACJA ===
MODEL_DIR = "model_llm"
os.makedirs(MODEL_DIR, exist_ok=True)
CONFIG_FILE = "modele.json"
N_THREADS = 4


def load_config():
    with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
        return json.load(f)


def get_memory_usage():
    """Zwraca aktualne użycie RAM przez proces w MB"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)


def run_benchmark(question, vectorstore):
    config = load_config()
    model_keys = list(config["models"].keys())

    print(f"🚀 Rozpoczynam benchmark dla {len(model_keys)} modeli.")

    for model_key in model_keys:
        print(f"\n{'=' * 20}")
        print(f" MODEL: {model_key}")
        print(f"{'=' * 20}")
        current_llm = get_llm(model_key, config)
        answer = ask_rag(question, vectorstore, current_llm, model_key, config)
        print(f"Odp: {answer}")
        del current_llm
        import gc
        gc.collect()
        time.sleep(2) 

    print("\n✅ Wszystkie modele zostały przetestowane.")


# Pomocnicze funkcje dostosowane do pętli:

def get_llm(model_key, config):
    model_cfg = config["models"][model_key]
    repo_id = model_cfg["repo_id"]
    filename = model_cfg["filename"]

    os.makedirs(MODEL_DIR, exist_ok=True)
    model_path = os.path.join(MODEL_DIR, filename)

    if not os.path.exists(model_path):
        print(f"Pobieranie {filename}...")
        hf_hub_download(repo_id=repo_id, filename=filename, local_dir=MODEL_DIR)

    mem_before = get_memory_usage()
    llm = Llama(
        model_path=model_path,
        n_ctx=2048,
        n_threads=N_THREADS,
        n_gpu_layers=0,
        verbose=False
    )
    mem_after = get_memory_usage()
    print(f"✓ Załadowano. Przyrost RAM: {mem_after - mem_before:.2f} MB")
    return llm


def ask_rag(question, vectorstore, llm, model_key, config):
    start_time = time.time()
    template = config["models"][model_key]["prompt_template"]

    docs = vectorstore.similarity_search(question, k=3)
    context = "\n".join([d.page_content for d in docs]) if docs else "Brak kontekstu."
    prompt = template.format(context=context, question=question)

    response = llm(prompt, max_tokens=200, temperature=0.0)
    answer = response['choices'][0]['text'].strip()

    elapsed = time.time() - start_time
    mem_end = get_memory_usage()

    save_result_to_json(model_key, question, answer, elapsed, mem_end)
    return answer


def save_result_to_json(model_name, question, answer, duration, ram_usage):
    """Logowanie odpowiedzi wraz z wydajnością"""
    res_file = "results_pdf.json"
    results = {}
    if os.path.exists(res_file):
        try:
            with open(res_file, 'r', encoding='utf-8') as f:
                results = json.load(f)
        except:
            results = {}

    entry_id = f"{model_name}_{int(time.time())}"
    results[entry_id] = {
        "model": model_name,
        "question": question,
        "answer": answer,
        "metrics": {
            "duration_sec": round(duration, 2),
            "ram_usage_mb": round(ram_usage, 2)
        }
    }

    with open(res_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=4, ensure_ascii=False)



#-------- MAIN -----------
# Stałe konfiguracyjne
COLLECTION_NAME = "documents_pdf"
DOKUMENTY_PATH = "dokumenty/pdfy"


def main():
    # 1. Inicjalizacja klienta bazy
    client = init_chroma()

    # 2. Sprawdzenie/Ładowanie bazy dokumentów
    if check_collection_exists(client, COLLECTION_NAME):
        print("✓ Baza danych gotowa. Ładowanie istniejącej kolekcji...")
        embeddings = get_embeddings()
        vectorstore = load_vectorstore(client, COLLECTION_NAME, embeddings)
    else:
        print("✗ Kolekcja nie istnieje. Tworzenie nowej...")
        docs = load_pdf_documents(DOKUMENTY_PATH)
        if not docs:
            print("Błąd: Brak dokumentów!")
            return
        
        all_splits = smart_chunk_documents(docs)
        embeddings = get_embeddings()
        vectorstore = index_documents_in_batches(all_splits, embeddings, client, COLLECTION_NAME)

    # 3. Definicja pytania testowego
    query = "Jakie ubezpieczenie uczelnia zaleca studentom?"
    print(f"👉 {query}\n")

    # 4. Wywołanie benchmarku (pętla po wszystkich modelach z JSON), Ta funkcja sama zajmie się ładowaniem, pytaniem i czyszczeniem RAM
    run_benchmark(query, vectorstore)

    print("\n📊 Wyniki zostały zapisane do pliku results_pdf.json")


if __name__ == "__main__":
    main()




✗ Kolekcja 'documents_pdf' nie istnieje
✗ Kolekcja nie istnieje. Tworzenie nowej...
--- Ładowanie PDFów ---
✓ Załadowano: 1.pdf
✓ Załadowano: 2.pdf
✓ Załadowano: 3.pdf
--- Chunkowanie dokumentów ---
Łączna liczba dokumentów: 97
Średnia długość dokumentu: 2239 znaków
✓ Wybieram strategię dla długich dokumentów akademickich
Utworzono 301 fragmentów
Średnia długość fragmentu: 728 znaków
--- Inicjalizacja modelu embeddingowego ---


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Mateusz\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\PYTHON\baza_wektorowa\RAG_TEST\model_embedding\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

--- Indeksowanie w ChromaDB ---
Zaindeksowano 50/301 fragmentów
Zaindeksowano 100/301 fragmentów
Zaindeksowano 150/301 fragmentów
Zaindeksowano 200/301 fragmentów
Zaindeksowano 250/301 fragmentów
Zaindeksowano 300/301 fragmentów
Zaindeksowano 301/301 fragmentów
⏱ Czas indeksowania: 355.31s
👉 Jakie ubezpieczenie uczelnia zaleca studentom?

🚀 Rozpoczynam benchmark dla 18 modeli.

 MODEL: gemma_9b
Pobieranie gemma-2-9b-it-IQ2_M.gguf...


llama_context: n_ctx_per_seq (2048) < n_ctx_train (8192) -- the full capacity of the model will not be utilized
llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


✓ Załadowano. Przyrost RAM: 642.35 MB


Poniżej znajduje się kompletny kod do uruchomienia RAG z porównaniem różnych modeli LLM. Kod jest podzielony na sekcje.
Tutaj poniżej jest **dodawanie danych z plików TXT do bazy wektorowej.**

In [ ]:
MODEL_EMBEDDING_NAME = "sentence-transformers/all-mpnet-base-v2"

# ------------------------------------------------------ Embedding ----------------------------------------------------------------
from langchain_huggingface import HuggingFaceEmbeddings
import os
MODEL_SAVE_PATH = "model_embedding"
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': True, 'batch_size': 8}
def get_embeddings():
    print("--- Inicjalizacja modelu embeddingowego ---")
    return HuggingFaceEmbeddings(
        model_name=MODEL_EMBEDDING_NAME,
        cache_folder=MODEL_SAVE_PATH,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs,
        multi_process=False
    )


# ----------------------------------------------------------- Chunkowanie -----------------------------------------

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    SentenceTransformersTokenTextSplitter
)
CHUNK_SIZE = 900
CHUNK_OVERLAP = 50
def create_optimized_splitter():
    """Splitter zoptymalizowany pod polski język akademicki"""
    return RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
        separators=["\n\n", "\n", ". ", "! ", "? ",  "; ",  ", ", " ",  ""]
    )
def create_token_splitter():
    """Splitter bazujący na tokenach - dokładniejszy"""
    return SentenceTransformersTokenTextSplitter(
        chunk_overlap=CHUNK_OVERLAP,
        tokens_per_chunk=256,
        model_name=MODEL_EMBEDDING_NAME
    )
def smart_chunk_documents(docs):
    """Inteligentne chunkowanie z diagnostyką"""
    print("--- Chunkowanie dokumentów ---")
    if not docs:
        print("Brak dokumentów do przetworzenia")
        return []
    total_chars = sum(len(doc.page_content) for doc in docs)
    avg_doc_length = total_chars / len(docs)
    print(f"Łączna liczba dokumentów: {len(docs)}")
    print(f"Średnia długość dokumentu: {avg_doc_length:.0f} znaków")
    if avg_doc_length > 1000:
        print("✓ Wybieram strategię dla długich dokumentów akademickich")
        splitter = create_optimized_splitter()
    else:
        print("✓ Wybieram strategię token-based dla lepszej precyzji")
        splitter = create_token_splitter()

    chunks = splitter.split_documents(docs)
    if chunks:
        avg_chunk_size = sum(len(chunk.page_content) for chunk in chunks) / len(chunks)
        print(f"Utworzono {len(chunks)} fragmentów")
        print(f"Średnia długość fragmentu: {avg_chunk_size:.0f} znaków")
    else:
        raise ValueError("Błąd: Nie udało się utworzyć fragmentów dokumentów")
    return chunks

# ----------------------------------------------------------- Dodawanie do bazy ------------------------------------------------------------------
import chromadb
import os
import time
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader

# === KONFIGURACJA ŚCIEZEK ===
DB_DIR = "bazawektorowa_txt"
os.makedirs(DB_DIR, exist_ok=True)

def init_chroma():
    os.makedirs(DB_DIR, exist_ok=True)
    return chromadb.PersistentClient(path=DB_DIR)

def check_collection_exists(client, collection_name):
    """Sprawdza czy kolekcja istnieje i zawiera dane"""
    try:
        collection = client.get_collection(collection_name)
        count = collection.count()
        if count > 0:
            print(f"✓ Kolekcja '{collection_name}' istnieje i zawiera {count} dokumentów")
            return True
        else:
            print(f"✗ Kolekcja '{collection_name}' istnieje ale jest pusta")
            return False
    except Exception as e:
        print(f"✗ Kolekcja '{collection_name}' nie istnieje")
        return False

def save_to_vectorstore(documents, embeddings, client, collection_name):
    """Zapis dokumentów do bazy wektorowej"""
    return Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        client=client,
        collection_name=collection_name,
    )

def load_vectorstore(client, collection_name, embeddings):
    print("--- Ładowanie bazy wektorowej ---")
    return Chroma(
        client=client,
        collection_name=collection_name,
        embedding_function=embeddings
    )

def load_txt_documents(documents_path):
    """Ładowanie dokumentów TXT z folderu"""
    print("--- Ładowanie plików TXT ---")
    docs = []
    if not os.path.exists(documents_path):
        return []
    for file in os.listdir(documents_path):
        if file.endswith(".txt"):
            try:
                loader = TextLoader(os.path.join(documents_path, file),"utf-8")
                docs.extend(loader.load())
                print(f"✓ Załadowano: {file}")
            except Exception as e:
                print(f"✗ Błąd z {file}: {e}")
    return docs



def index_documents_in_batches(splits, embeddings, client, collection_name, batch_size=50):
    print("--- Indeksowanie w ChromaDB ---")
    start_time = time.time()
    vectorstore = None

    for i in range(0, len(splits), batch_size):
        batch = splits[i:i + batch_size]
        vectorstore = save_to_vectorstore(batch, embeddings, client, collection_name)
        print(f"Zaindeksowano {min(i + batch_size, len(splits))}/{len(splits)} fragmentów")

    print(f"⏱ Czas indeksowania: {time.time() - start_time:.2f}s")
    return vectorstore



# ------------------------------------------------------------- INICJALIZACJA RAG --------------------------------------------------------------------

import json
import os
import time
import psutil 
from llama_cpp import Llama
from huggingface_hub import hf_hub_download
from tqdm import tqdm


# === KONFIGURACJA ===
MODEL_DIR = "model_llm"
os.makedirs(MODEL_DIR, exist_ok=True)
CONFIG_FILE = "modele.json"
N_THREADS = 4


def load_config():
    with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
        return json.load(f)


def get_memory_usage():
    """Zwraca aktualne użycie RAM przez proces w MB"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)


def run_benchmark(question, vectorstore):
    config = load_config()
    model_keys = list(config["models"].keys())

    print(f"🚀 Rozpoczynam benchmark dla {len(model_keys)} modeli.")

    for model_key in model_keys:
        print(f"\n{'=' * 20}")
        print(f"🤖 MODEL: {model_key}")
        print(f"{'=' * 20}")
        current_llm = get_llm(model_key, config)
        answer = ask_rag(question, vectorstore, current_llm, model_key, config)
        print(f"Odp: {answer}")
        # 3. CZYSZCZENIE PAMIĘCI
        del current_llm
        import gc
        gc.collect()
        time.sleep(2)
    print("\n✅ Wszystkie modele zostały przetestowane.")

def get_llm(model_key, config):
    model_cfg = config["models"][model_key]
    repo_id = model_cfg["repo_id"]
    filename = model_cfg["filename"]

    os.makedirs(MODEL_DIR, exist_ok=True)
    model_path = os.path.join(MODEL_DIR, filename)

    if not os.path.exists(model_path):
        print(f"Pobieranie {filename}...")
        hf_hub_download(repo_id=repo_id, filename=filename, local_dir=MODEL_DIR)

    mem_before = get_memory_usage()
    llm = Llama(
        model_path=model_path,
        n_ctx=2048,
        n_threads=N_THREADS,
        n_gpu_layers=0,
        verbose=False
    )
    mem_after = get_memory_usage()
    print(f"✓ Załadowano. Przyrost RAM: {mem_after - mem_before:.2f} MB")
    return llm


def ask_rag(question, vectorstore, llm, model_key, config):
    start_time = time.time()
    template = config["models"][model_key]["prompt_template"]

    docs = vectorstore.similarity_search(question, k=3)
    context = "\n".join([d.page_content for d in docs]) if docs else "Brak kontekstu."
    prompt = template.format(context=context, question=question)

    response = llm(prompt, max_tokens=200, temperature=0.0)
    answer = response['choices'][0]['text'].strip()

    elapsed = time.time() - start_time
    mem_end = get_memory_usage()

    save_result_to_json(model_key, question, answer, elapsed, mem_end)
    return answer


def save_result_to_json(model_name, question, answer, duration, ram_usage):
    """Logowanie odpowiedzi wraz z wydajnością"""
    res_file = "results_txt_.json"
    results = {}
    if os.path.exists(res_file):
        try:
            with open(res_file, 'r', encoding='utf-8') as f:
                results = json.load(f)
        except:
            results = {}

    entry_id = f"{model_name}_{int(time.time())}"
    results[entry_id] = {
        "model": model_name,
        "question": question,
        "answer": answer,
        "metrics": {
            "duration_sec": round(duration, 2),
            "ram_usage_mb": round(ram_usage, 2)
        }
    }

    with open(res_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=4, ensure_ascii=False)



#------------------------------------------ MAIN ---------------------------------------------------------

# Stałe konfiguracyjne
COLLECTION_NAME = "documents_txt"
DOKUMENTY_PATH = "dokumenty/txt"


def main():
    client = init_chroma()
    if check_collection_exists(client, COLLECTION_NAME):
        print("✓ Baza danych gotowa. Ładowanie istniejącej kolekcji...")
        embeddings = get_embeddings()
        vectorstore = load_vectorstore(client, COLLECTION_NAME, embeddings)
    else:
        print("✗ Kolekcja nie istnieje. Tworzenie nowej...")
        docs = load_txt_documents(DOKUMENTY_PATH)
        if not docs:
            print("Błąd: Brak dokumentów!")
            return
        
        all_splits = smart_chunk_documents(docs)
        embeddings = get_embeddings()
        vectorstore = index_documents_in_batches(all_splits, embeddings, client, COLLECTION_NAME)

    # 3. Definicja pytania testowego
    query = "Jakie ubezpieczenie uczelnia zaleca studentom?"
    print(f"👉 {query}\n")

    # 4. Wywołanie benchmarku (pętla po wszystkich modelach z JSON) Ta funkcja sama zajmie się ładowaniem, pytaniem i czyszczeniem RAM
    run_benchmark(query, vectorstore)

    print("\n📊 Wyniki zostały zapisane do pliku results_txt.json")


if __name__ == "__main__":
    main()

